# Домашнее задание № 3
## 1. EDA по датасету [Iris Species](https://www.kaggle.com/datasets/uciml/iris)

In [38]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv('Iris.csv')
df.shape

(150, 6)

In [ ]:
df.dtypes

In [39]:
df.isnull().sum()

Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64

In [ ]:
df.value_counts(subset=['Species'])

In [ ]:
df['Id'].duplicated().sum()

In [ ]:
df[df['SepalLengthCm'] <= 0].count()

In [ ]:
df[df['PetalLengthCm'] <= 0].count()

In [ ]:
df[df['PetalWidthCm'] <= 0].count()

In [ ]:
df[df['SepalWidthCm'] <= 0].count()

In [ ]:
df['PetalLengthCm'].mode()

In [ ]:
df['PetalLengthCm'].mode()

## 2. Подготовка данных

In [ ]:
df_enc = pd.get_dummies(df, columns=['Species'])
species_cols = ['Species_Iris-setosa', 'Species_Iris-versicolor', 'Species_Iris-virginica']
df_enc['Species'] = df_enc[species_cols].values.argmax(axis=1)
df_enc = df_enc.drop(columns=species_cols)
df_enc.head(60)

In [ ]:
X = df_enc[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']].values
Y = df_enc['Species'].values

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

### Выборка из масштабированных признаков

In [32]:
scaler = StandardScaler()
X_sc = scaler.fit_transform(df_enc[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']].values) 
X_train_sc,  X_test_sc, Y_train_sc, Y_test_sc = train_test_split(X_sc, Y, test_size=0.2, random_state=42)

Из-за того что KNN основан на расстояниях, важно сделать масштабирование, если хотим, чтобы все признаки были равноправными, и не было ситуаций, что признак, который больше по значению, больше влиял на прогноз алгоритма

Подбор параметров на тестовой выборке крайне нежелателен, так как модель может показывать завышенные результаты, и шанс её переобучения достаточно высок. Тестовую выборку нужно использовать только 1 раз - для финальной оценки модели



## Реализация KNN

In [31]:
results = []
k_values = [1, 3, 5, 11, 17, 27]
metrics = ['euclidean', 'manhattan', 'minkowski', 'chebyshev', 'cosine']
weights = ['uniform', 'distance']

for k in k_values:
    for metric in metrics:
        for weight in weights:
            p = 2
            final_metric = metric
            if metric == 'euclidean':
                final_metric = 'minkowski'
            elif metric == 'manhattan':
                p = 1
                final_metric = 'minkowski'
            elif metric == 'minkowski':
                p = 3
            else:
                p = None
            knn = KNeighborsClassifier(n_neighbors=k, weights=weight, metric=final_metric, p=p)
            knn_scaled = KNeighborsClassifier(n_neighbors=k, weights=weight, metric=final_metric, p=p)
            knn.fit(X_train, Y_train)
            knn_scaled.fit(X_train_sc, Y_train_sc)
            acc = knn.score(X_test, Y_test)
            acc_scaled = knn_scaled.score(X_test_sc, Y_test_sc)
            results.append({
                'k': k,
                'metric': metric,
                'weights': weight,
                'accuracy without features scaling': acc,
                'accuracy with features scaling': acc_scaled,
            })
            
results_df = pd.DataFrame(results)
results_df

,k,metric,weights,accuracy without features scaling,accuracy with features scaling
0,1,euclidean,uniform,1.000000,0.966667
1,1,euclidean,distance,1.000000,0.966667
2,1,manhattan,uniform,1.000000,0.966667
3,1,manhattan,distance,1.000000,0.966667
4,1,minkowski,uniform,1.000000,1.000000
5,1,minkowski,distance,1.000000,1.000000
6,1,chebyshev,uniform,1.000000,1.000000
7,1,chebyshev,distance,1.000000,1.000000
8,1,cosine,uniform,0.966667,0.900000
9,1,cosine,distance,0.966667,0.900000


## Подбор гиперпараметров

In [37]:
results = []
for k in k_values:
    for metric in metrics:
        for weight in weights:
            p = 2
            final_metric = metric
            if metric == 'euclidean':
                final_metric = 'minkowski'
            elif metric == 'manhattan':
                p = 1
                final_metric = 'minkowski'
            elif metric == 'minkowski':
                p = 3
            else:
                p = None
            knn = KNeighborsClassifier(n_neighbors=k, weights=weight, metric=final_metric, p=p)
            knn.fit(X_train, Y_train)
            cv_scores = cross_val_score(knn, X_train, Y_train, cv=5, scoring='accuracy')
            acc = knn.score(X_test, Y_test)
            
            results.append({
                'k': k,
                'metric': metric,
                'weights': weight,
                'score_mean': cv_scores.std(),
                'accuracy': acc

            })
            
results_df = pd.DataFrame(results)
results_df

,k,metric,weights,score_mean,accuracy
0,1,euclidean,uniform,0.040825,1.000000
1,1,euclidean,distance,0.040825,1.000000
2,1,manhattan,uniform,0.040825,1.000000
3,1,manhattan,distance,0.040825,1.000000
4,1,minkowski,uniform,0.040825,1.000000
5,1,minkowski,distance,0.040825,1.000000
6,1,chebyshev,uniform,0.048591,1.000000
7,1,chebyshev,distance,0.048591,1.000000
8,1,cosine,uniform,0.037268,0.966667
9,1,cosine,distance,0.037268,0.966667
